# 01 — EDA: Diabetes 130-US hospitals

Lightweight exploration of the **cleaned** dataset (`data/processed/clean.parquet`).
Goal: sanity-check distributions before feature engineering.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.data.clean import CLEAN_PARQUET, TARGET_BIN

df = pd.read_parquet(CLEAN_PARQUET)
df.shape


## Target balance


In [ ]:
df[TARGET_BIN].value_counts(normalize=True).round(4)


## Numeric columns — describe


In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
df[num_cols].describe().T


## Categorical cardinalities


In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
card = df[cat_cols].nunique().sort_values(ascending=False)
card.to_frame('n_unique')


## Key distributions


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['age'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='age')
df['gender'].value_counts().plot(kind='bar', ax=axes[1], title='gender')
df['race'].value_counts().plot(kind='bar', ax=axes[2], title='race')
plt.tight_layout()
plt.show()


## Target vs key categoricals (base rate check)


In [ ]:
for col in ['age', 'gender', 'race']:
    print(f'--- {col} ---')
    print(df.groupby(col)[TARGET_BIN].agg(['mean', 'size']).round(4))
    print()


## Notes to carry over to Phase 4 (feature engineering)

- `age` is a 10-year bucket string -> ordinal encode.
- `diag_1/2/3` are ICD-9 codes -> group into top-N + 'Other'.
- 11 % positive class -> keep PR-AUC as primary metric.
